In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import base64
import os
import sys
from uuid import uuid4

from cryptography.hazmat.primitives import serialization
from pyspark.sql import SparkSession, functions as F

import snowflake.connector
from dotenv import dotenv_values

In [2]:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["JAVA_HOME"] = (
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
)
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 512m pyspark-shell"

spark = (
    SparkSession.builder
    .appName("SkyPoints")
    .master("local[1]")
    .config("spark.sql.shuffle.partitions", "2")

    # Load the S3 connector and Snowflake Spark/JDBC connectors.
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "net.snowflake:spark-snowflake_2.12:3.2.0-spark_3.5,"
        "net.snowflake:snowflake-jdbc:4.0.2",
    )

    # Locate MinIO.
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:19000")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    # Authenticate to MinIO.
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "admin123")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)

26/11/23 03:49:36 WARN Utils: Your hostname, Monisankars-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.5 instead (on interface en0)
26/11/23 03:49:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/mskbaruah/.ivy2/cache
The jars for the packages stored in: /Users/mskbaruah/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
net.snowflake#spark-snowflake_2.12 added as a dependency
net.snowflake#snowflake-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4ff33a0c-37cf-468e-84c4-2eaaa9b35f26;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/mskbaruah/code/Incubyte%20Take%20home%20assignment/skypoints-pipeline/skypoints/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found net.snowflake#spark-snowflake_2.12;3.2.0-spark_3.5 in central
	found net.snowflake#snowflake-jdbc;4.0.2 in central
	found org.apache.parquet#parquet-avro;1.15.2 in central
	found org.apache.parquet#parquet-column;1.15.2 in central
	found org.apache.parquet#parquet-common;1.15.2 in central
	found org.apache.parquet#parquet-format-structures;1.15.2 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.slf4j#slf4j-api;1.7.33 in central
	found org.apache.parquet#parquet-encoding;1.15.2 in central
	found org.apache.parquet#parquet-hadoop;1.15.2 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found io.airlift#aircompressor;2.0.2 in central
	found commons-pool#commons-pool;1.6 in central
	found com.github.luben#zstd-jni;1.5.6-6 in central
	found org.apache.av

In [3]:
from pyspark.sql import functions as F

landing_path = "s3a://skypoints-landing/"

# Read every matching member file into one DataFrame.
members_raw = (
    spark.read
    .option("pathGlobFilter", "members_*.txt")
    .text(landing_path)
    .withColumn("source_file", F.input_file_name())
)

# Read every matching redemption file into another DataFrame.
redemptions_raw = (
    spark.read
    .option("pathGlobFilter", "redemptions_*.json")
    .option("multiLine", True)
    .json(landing_path)
    .withColumn("source_file", F.input_file_name())
)

members_raw.show(5, truncate = False)
redemptions_raw.show(5, truncate = False)

26/11/23 03:49:49 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+----------------------------------------------------------------------------------------------------------+----------------------------------------+
|value                                                                                                     |source_file                             |
+----------------------------------------------------------------------------------------------------------+----------------------------------------+
||H|Member_Name|Member_Id|Enrollment_Date|Last_Flight_Date|Tier_Code|Agent_Name|State|Country|DOB|Is_Active|s3a://skypoints-landing/members_0001.txt|
||D|Monserrate Johns|1|20230211|20260922|SLV|Olivia Hart|03|LCA|04011995|I                                 |s3a://skypoints-landing/members_0001.txt|
||D|Hector Heller|2|20231002|20260708|BLU|Maya Arden|02|ARE|13061978|A                                     |s3a://skypoints-landing/members_0001.txt|
||D|Keshaun Lang|3|20180619|20260830|SLV|Leah Nolan|02|PRT|07041995|I                               

In [4]:
# 1. Read the saved Snowflake credentials.
sf = dotenv_values(
    "/Users/mskbaruah/code/Incubyte Take home assignment/"
    "skypoints-pipeline/.env.local",
    interpolate=False,
)

# 2. Prepare the encrypted private key for the Spark connector.
passphrase = sf.get("SNOWFLAKE_PRIVATE_KEY_PASSPHRASE")

private_key = serialization.load_pem_private_key(
    open(sf["SNOWFLAKE_LOCAL_PRIVATE_KEY_FILE"], "rb").read(),
    password=passphrase.encode() if passphrase else None,
)

key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)


In [5]:
# Contains private-key material; do not display this dictionary.
sf_options = {
    "sfURL": f"{sf['SNOWFLAKE_ACCOUNT']}.snowflakecomputing.com",
    "sfUser": sf["SNOWFLAKE_USER"],
    "sfRole": sf["SNOWFLAKE_ROLE"],
    "sfWarehouse": sf["SNOWFLAKE_WAREHOUSE"],
    "sfDatabase": sf["SNOWFLAKE_DATABASE"],
    "sfSchema": "RAW",
    "pem_private_key": base64.b64encode(key_bytes).decode("ascii"),
}

In [6]:
# 3. Verify that Spark can connect to Snowflake.
(
    spark.read
    .format("net.snowflake.spark.snowflake")
    .options(**sf_options)
    .option("query", "SELECT 1 AS CONNECTION_OK")
    .load()
    .show()
)


26/11/23 03:54:06 ERROR RestRequest: Stop retrying since elapsed time due to network issues has reached timeout. Elapsed: 1,017 ms, timeout: 1,000 ms
26/11/23 03:54:06 ERROR RestRequest: Stop retrying since elapsed time due to network issues has reached timeout. Elapsed: 1,017 ms, timeout: 1,000 ms
26/11/23 03:54:06 ERROR RestRequest: Returning null response. Cause: java.net.UnknownHostException: metadata.google.internal, request: GET http://metadata.google.internal HTTP/1.1
26/11/23 03:54:06 ERROR RestRequest: Returning null response. Cause: java.net.UnknownHostException: metadata.google.internal, request: GET http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/email HTTP/1.1
26/11/23 03:54:06 ERROR RestRequest: Stop retrying since elapsed time due to network issues has reached timeout. Elapsed: 1,413 ms, timeout: 1,000 ms
26/11/23 03:54:06 ERROR RestRequest: Stop retrying since elapsed time due to network issues has reached timeout. Elapsed: 1,414 ms,

+-------------+
|CONNECTION_OK|
+-------------+
|            1|
+-------------+



In [7]:
# 4. Prepare the raw records and their source filenames.
members_to_write = members_raw.select(
    F.col("value").alias("RAW_RECORD"),
    (
        F.col("source_file")
        if "source_file" in members_raw.columns
        else F.input_file_name()
    ).alias("SOURCE_FILE"),
)

payload_columns = [
    name for name in redemptions_raw.columns
    if name != "source_file"
]

redemptions_to_write = redemptions_raw.select(
    F.struct(*payload_columns).alias("RAW_PAYLOAD"),
    (
        F.col("source_file")
        if "source_file" in redemptions_raw.columns
        else F.input_file_name()
    ).alias("SOURCE_FILE"),
)

In [ ]:
# 5. Write both feeds with batch and ingestion metadata.
batch_id = uuid4().hex

feeds = {
    "MEMBERS_RAW": members_to_write,
    "REDEMPTIONS_RAW": redemptions_to_write,
}

for table_name, source_df in feeds.items():

    output_df = (
        source_df
        .withColumn("BATCH_ID", F.lit(batch_id))
        .withColumn("INGESTED_AT", F.current_timestamp())
    )

    writer = (
        output_df.write
        .format("net.snowflake.spark.snowflake")
        .options(**sf_options)
        .option("dbtable", table_name)
        .option("column_mapping", "name")
        .mode("overwrite")
    )

    print(f"Writing {table_name}...")
    writer.save()
    print(f"Loaded {table_name}")

Writing MEMBERS_RAW...


26/11/23 04:21:56 WARN VersionInfoUtils: The AWS SDK for Java 1.x entered maintenance mode starting July 31, 2024 and will reach end of support on December 31, 2025. For more information, see https://aws.amazon.com/blogs/developer/the-aws-sdk-for-java-1-x-is-in-maintenance-mode-effective-july-31-2024/
You can print where on the file system the AWS SDK for Java 1.x core runtime is located by setting the AWS_JAVA_V1_PRINT_LOCATION environment variable or aws.java.v1.printLocation system property to 'true'.
This message can be disabled by setting the AWS_JAVA_V1_DISABLE_DEPRECATION_ANNOUNCEMENT environment variable or aws.java.v1.disableDeprecationAnnouncement system property to 'true'.
The AWS SDK for Java 1.x is being used here:
at java.base/java.lang.Thread.getStackTrace(Thread.java:1619)
at net.snowflake.spark.shaded.amazonaws.util.VersionInfoUtils.printDeprecationAnnouncement(VersionInfoUtils.java:81)
at net.snowflake.spark.shaded.amazonaws.util.VersionInfoUtils.<clinit>(VersionInfoU

Loaded MEMBERS_RAW
Writing REDEMPTIONS_RAW...


26/11/23 04:22:02 WARN LogicalTypes: Ignoring invalid logical type for name: timestamp-micros
26/11/23 04:22:02 WARN CloudStorageOperations$: Discarding partial file 0.PARQUET.gz. success=false, isInterrupted=false
26/11/23 04:22:02 WARN CloudStorageOperations$: Spark Connector Worker: hit upload error: partition ID:0 0.PARQUET.gz attemptNumber=0 Skip exponential backoff sleep because use_exponential_backoff is 'off'. Please enable it if necessary, for example, cloud service throttling issues happen.
26/11/23 04:22:02 ERROR Executor: Exception in task 0.0 in stage 5.0 (TID 5)
java.lang.NoSuchMethodError: 'org.apache.hadoop.conf.Configuration org.apache.parquet.hadoop.util.ConfigurationUtil.createHadoopConfiguration(org.apache.parquet.conf.ParquetConfiguration)'
	at org.apache.parquet.avro.AvroWriteSupport.getDataModel(AvroWriteSupport.java:436)
	at org.apache.parquet.avro.AvroWriteSupport.init(AvroWriteSupport.java:141)
	at org.apache.parquet.avro.AvroWriteSupport.init(AvroWriteSupport

Py4JJavaError: An error occurred while calling o161.save.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 5.0 failed 1 times, most recent failure: Lost task 0.0 in stage 5.0 (TID 5) (192.168.1.5 executor driver): java.lang.NoSuchMethodError: 'org.apache.hadoop.conf.Configuration org.apache.parquet.hadoop.util.ConfigurationUtil.createHadoopConfiguration(org.apache.parquet.conf.ParquetConfiguration)'
	at org.apache.parquet.avro.AvroWriteSupport.getDataModel(AvroWriteSupport.java:436)
	at org.apache.parquet.avro.AvroWriteSupport.init(AvroWriteSupport.java:141)
	at org.apache.parquet.avro.AvroWriteSupport.init(AvroWriteSupport.java:130)
	at org.apache.parquet.hadoop.ParquetWriter.<init>(ParquetWriter.java:279)
	at org.apache.parquet.hadoop.ParquetWriter$Builder.build(ParquetWriter.java:702)
	at net.snowflake.spark.snowflake.io.CloudStorage.doUploadPartition(CloudStorageOperations.scala:722)
	at net.snowflake.spark.snowflake.io.CloudStorage.uploadPartiti